# 06 – RRF, Agregasi ke Lagu, dan Evaluasi

Masukan: indeks BM25 dan TF-IDF karakter (notebook 03), embedding (04), kueri sintetis (05), tabel chunk (02).

Alur:
1. Tiap kueri dicari oleh tiga retriever (BM25, TF-IDF karakter, dense), masing-masing mengembalikan 100 chunk teratas.
   Hasilnya disimpan (*cache*) agar variasi fusi bisa dievaluasi tanpa mencari ulang.
2. **RRF** pada tingkat chunk (persamaan 5), lalu **agregasi max** ke tingkat lagu (persamaan 6).
3. Parameter RRF (k dan kedalaman daftar) disetel pada **kueri validasi**; hasil akhir dilaporkan pada **kueri uji**.
4. Metrik: Recall@1/5/10, MRR, nDCG@10 (satu lagu relevan), interval kepercayaan bootstrap, uji berpasangan, latensi p50/p95, ukuran indeks.

Catatan: k1 dan b BM25 memakai nilai umum (1,2 dan 0,75) dan **tidak disetel**, karena bobot BM25 sudah tertanam di indeks.

In [1]:
# Konfigurasi dan data
import os, sys, glob, gc, json, time, math
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy import stats
import multiprocessing as mp
from sklearn.feature_extraction.text import HashingVectorizer
from tqdm.auto import tqdm

sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/datasets/nshamid/lyrecall-songs-clean/lyrecall_text.py", recursive=True)[0]))
from lyrecall_text import normalize_lex, clean_lyrics

def find(name):
    return glob.glob(f"/kaggle/input/**/{name}", recursive=True)[0]

# ====== KONFIGURASI ======
SEED      = 42
K_CAND    = 100      # daftar kandidat chunk per retriever
EVAL_TOP  = 100      # panjang daftar lagu yang dinilai
VARIANTS  = ["exact", "partial", "noisy"]     # tambah "typo", "sound" untuk analisis tambahan (lebih lama)
N_PROC    = os.cpu_count()
OUT_DIR   = "/kaggle/working"

chunks = pd.read_parquet(find("chunks_L3_S2.parquet"), columns=["chunk_id", "song_id"])
song_of = chunks["song_id"].to_numpy()
N = len(chunks)
Q = pd.read_parquet(find("queries.parquet")).reset_index(drop=True)
target = Q["song_id"].to_numpy()
val_idx = np.flatnonzero(Q["split"] == "val")
test_idx = np.flatnonzero(Q["split"] == "test")
print(f"Chunk: {N:,} | kueri: {len(Q):,} (val {len(val_idx):,}, uji {len(test_idx):,}) | CPU: {N_PROC}")

Chunk: 1,843,304 | kueri: 4,000 (val 1,000, uji 3,000) | CPU: 4


In [2]:
# Indeks leksikal dan fungsi pencarian
lp = json.load(open(find("lexical_params.json")))
assert lp["n_chunks"] == N, "indeks leksikal dan tabel chunk tidak cocok"
bm25   = sp.load_npz(find("bm25_csc.npz"))
tfidf  = sp.load_npz(find("tfidf_csc.npz"))
idf_c  = np.load(find("tfidf_idf.npy"))
vocab  = json.load(open(find("bm25_vocab.json")))
SUBLINEAR = lp["sublinear_tf"]
hv = HashingVectorizer(analyzer="char", ngram_range=tuple(lp["ngram"]), n_features=lp["hash_features"],
                       alternate_sign=False, norm=None, lowercase=False, dtype=np.float32)
print("BM25:", bm25.shape, "| TF-IDF:", tfidf.shape)

def score_postings(M, ids, weights, n_docs):
    s = np.zeros(n_docs, dtype=np.float32)
    ip, ix, dt = M.indptr, M.indices, M.data
    for j, w in zip(ids, weights):
        a, e = ip[j], ip[j + 1]
        s[ix[a:e]] += w * dt[a:e]
    return s

def topk(scores, k):
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    keep = scores[idx] > 0
    return idx[keep], scores[idx][keep]

EMPTY = (np.array([], dtype=int), np.array([], dtype=np.float32))

def search_bm25(q, k=100):
    ids = [vocab[t] for t in set(normalize_lex(q).split()) if t in vocab]
    if not ids:
        return EMPTY
    return topk(score_postings(bm25, ids, np.ones(len(ids), np.float32), N), k)

def search_tfidf(q, k=100):
    qv = hv.transform([normalize_lex(q)])
    ids = qv.indices
    if len(ids) == 0:
        return EMPTY
    w = qv.data.astype(np.float32)
    if SUBLINEAR:
        w = 1 + np.log(w)
    w = w * idf_c[ids]
    n = np.linalg.norm(w)
    if n == 0:
        return EMPTY
    return topk(score_postings(tfidf, ids, w / n, N), k)

def _lex_task(args):
    which, texts = args
    fn = search_bm25 if which == "bm25" else search_tfidf
    ids = np.full((len(texts), K_CAND), -1, np.int32)
    sc = np.zeros((len(texts), K_CAND), np.float32)
    for r, t in enumerate(texts):
        i, s = fn(t, K_CAND)
        ids[r, :len(i)] = i
        sc[r, :len(s)] = s
    return ids, sc

BM25: (1843304, 155138) | TF-IDF: (1843304, 4194304)


In [3]:
# Pencarian leksikal paralel
cache = {}
t0 = time.time()
ctx = mp.get_context("fork")        # proses anak berbagi memori indeks (tanpa salinan)
with ctx.Pool(N_PROC) as pool:
    for which in ["bm25", "tfidf"]:
        for v in VARIANTS:
            texts = Q["q_" + v].tolist()
            tasks = [(which, texts[i:i + 50]) for i in range(0, len(texts), 50)]
            parts = list(tqdm(pool.imap(_lex_task, tasks), total=len(tasks), desc=f"{which}-{v}"))
            cache[(which, v)] = (np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts]))
print(f"Pencarian leksikal selesai: {time.time() - t0:.0f} dtk")

def save_cache(path):
    np.savez(path, **{f"{r}|{v}|ids": a[0] for (r, v), a in cache.items()},
             **{f"{r}|{v}|sc": a[1] for (r, v), a in cache.items()})
save_cache(f"{OUT_DIR}/retrieval_cache.npz")

bm25-exact:   0%|          | 0/80 [00:00<?, ?it/s]

bm25-partial:   0%|          | 0/80 [00:00<?, ?it/s]

bm25-noisy:   0%|          | 0/80 [00:00<?, ?it/s]

tfidf-exact:   0%|          | 0/80 [00:00<?, ?it/s]

tfidf-partial:   0%|          | 0/80 [00:00<?, ?it/s]

tfidf-noisy:   0%|          | 0/80 [00:00<?, ?it/s]

Pencarian leksikal selesai: 1477 dtk


In [4]:
# Pencarian dense
import torch
from sentence_transformers import SentenceTransformer

ep = json.load(open(find("embed_params.json")))
assert ep["n_chunks"] == N, "embedding dan tabel chunk tidak cocok"
model = SentenceTransformer(ep["model"], device="cuda")
model.max_seq_length = ep["max_seq_length"]
model.half()
E = torch.from_numpy(np.load(find("chunk_emb_fp16.npy"))).to("cuda")      # N x 384, fp16
print("GPU:", torch.cuda.get_device_name(0), "| embedding:", tuple(E.shape))

def to_dense_text(t):
    return " ".join(clean_lyrics(t).split("\n"))

@torch.no_grad()
def dense_batch(texts, k=K_CAND, bs=128):
    ids_out, sc_out = [], []
    for i in range(0, len(texts), bs):
        qe = model.encode([to_dense_text(t) for t in texts[i:i + bs]], batch_size=bs,
                          convert_to_tensor=True, normalize_embeddings=True).to(E.dtype)
        v, ix = torch.topk(qe @ E.T, k, dim=1)
        ids_out.append(ix.cpu().numpy().astype(np.int32))
        sc_out.append(v.float().cpu().numpy())
    return np.concatenate(ids_out), np.concatenate(sc_out)

t0 = time.time()
for v in VARIANTS:
    cache[("dense", v)] = dense_batch(Q["q_" + v].tolist())
print(f"Pencarian dense selesai: {time.time() - t0:.0f} dtk")
save_cache(f"{OUT_DIR}/retrieval_cache.npz")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

GPU: Tesla T4 | embedding: (1843304, 384)
Pencarian dense selesai: 6 dtk


## Fusi RRF, agregasi ke lagu, dan metrik
- Satu retriever: peringkat lagu = urutan kemunculan pertama lagu pada daftar chunk (sama dengan skor maksimum chunk-nya).
- Fusi: skor RRF(c) = Σ 1/(k + rank) pada tingkat chunk, lalu skor lagu = max skor chunk-nya.
- Lagu yang tidak muncul di 100 teratas dihitung gagal (reciprocal rank = 0).
- nDCG@10 dengan satu lagu relevan: 1/log2(rank + 1) bila rank ≤ 10.

In [5]:
# Fungsi fusi dan metrik
def ranks_single(ids):
    """Urutan lagu dari satu daftar chunk (kemunculan pertama = skor chunk terbaik, persamaan 6)."""
    seen, out = set(), []
    for c in ids:
        if c < 0:
            break
        s = song_of[c]
        if s not in seen:
            seen.add(s); out.append(s)
    return out

def fuse(lists, depth, k):
    """RRF pada tingkat chunk (persamaan 5), lalu agregasi max ke lagu (persamaan 6)."""
    sc = {}
    for ids in lists:
        for r in range(min(depth, len(ids))):
            c = ids[r]
            if c < 0:
                break
            sc[c] = sc.get(c, 0.0) + 1.0 / (k + r + 1)
    best = {}
    for c, s in sc.items():
        sid = song_of[c]
        if s > best.get(sid, 0.0):
            best[sid] = s
    return [sid for sid, _ in sorted(best.items(), key=lambda x: -x[1])[:EVAL_TOP]]

def target_rank(song_list, tgt):
    try:
        return song_list.index(tgt) + 1
    except ValueError:
        return np.inf

def metrics(r):
    r = np.asarray(r, dtype=float)
    return {"R@1": np.mean(r <= 1), "R@5": np.mean(r <= 5), "R@10": np.mean(r <= 10),
            "MRR": np.mean(1.0 / r), "nDCG@10": np.mean(np.where(r <= 10, 1.0 / np.log2(r + 1), 0.0))}

METHODS = {
    "BM25":                     ["bm25"],
    "TF-IDF karakter":          ["tfidf"],
    "Dense (MiniLM)":           ["dense"],
    "RRF BM25+TF-IDF":          ["bm25", "tfidf"],
    "RRF BM25+Dense":           ["bm25", "dense"],
    "RRF TF-IDF+Dense":         ["tfidf", "dense"],
    "Hybrid RRF (3 retriever)": ["bm25", "tfidf", "dense"],
}
HYBRID = "Hybrid RRF (3 retriever)"

def eval_method(retrs, variant, rows, depth, k):
    out = np.empty(len(rows))
    for j, qi in enumerate(rows):
        lists = [cache[(r, variant)][0][qi] for r in retrs]
        lst = ranks_single(lists[0]) if len(lists) == 1 else fuse(lists, depth, k)
        out[j] = target_rank(lst, target[qi])
    return out

In [6]:
# Penyetelan RRF pada data validasi
grid_k, grid_depth = [10, 30, 60, 100], [20, 50, 100]
tune = []
for k in grid_k:
    for depth in grid_depth:
        row = {"k": k, "depth": depth}
        for v in VARIANTS:
            row["MRR_" + v] = metrics(eval_method(METHODS[HYBRID], v, val_idx, depth, k))["MRR"]
        row["MRR_rerata"] = np.mean([row["MRR_" + v] for v in VARIANTS])
        tune.append(row)
tune = pd.DataFrame(tune).sort_values("MRR_rerata", ascending=False).reset_index(drop=True)
display(tune.round(4))
RRF_K, DEPTH = int(tune.loc[0, "k"]), int(tune.loc[0, "depth"])
print(f"Dipilih dari data validasi: k = {RRF_K}, kedalaman = {DEPTH}  (nilai awal laporan: k = 60)")
tune.to_csv(f"{OUT_DIR}/rrf_tuning_val.csv", index=False)

,k,depth,MRR_exact,MRR_partial,MRR_noisy,MRR_rerata
0,10,100,0.9875,0.9402,0.9583,0.9620
1,10,50,0.9874,0.9408,0.9573,0.9618
2,10,20,0.9863,0.9408,0.9551,0.9608
3,30,20,0.9827,0.9368,0.9504,0.9566
4,100,20,0.9827,0.9365,0.9504,0.9565
5,60,20,0.9827,0.9365,0.9504,0.9565
6,30,50,0.9786,0.9323,0.9498,0.9536
7,30,100,0.9779,0.9307,0.9518,0.9534
8,60,50,0.9770,0.9289,0.9471,0.9510
9,100,50,0.9768,0.9283,0.9467,0.9506


Dipilih dari data validasi: k = 10, kedalaman = 100  (nilai awal laporan: k = 60)


In [7]:
# Evaluasi akhir pada data uji
rng = np.random.default_rng(SEED)
def boot_ci(x, B=2000):
    idx = rng.integers(0, len(x), size=(B, len(x)))
    return np.percentile(x[idx].mean(axis=1), [2.5, 97.5])

RANKS, rows = {}, []
for name, retrs in METHODS.items():
    for v in VARIANTS:
        r = eval_method(retrs, v, test_idx, DEPTH, RRF_K)
        RANKS[(name, v)] = r
        m = metrics(r)
        lo, hi = boot_ci(1.0 / r)
        rows.append({"method": name, "variant": v, **m, "MRR_lo": lo, "MRR_hi": hi})
res = pd.DataFrame(rows)
res.to_csv(f"{OUT_DIR}/results_test.csv", index=False)

for metric in ["R@1", "R@10", "MRR", "nDCG@10"]:
    print(f"\n== {metric} (data uji, n = {len(test_idx)}) ==")
    display(res.pivot(index="method", columns="variant", values=metric)[VARIANTS].loc[list(METHODS)].round(3))


== R@1 (data uji, n = 3000) ==


variant,exact,partial,noisy
method,,,
BM25,0.982,0.940,0.918
TF-IDF karakter,0.968,0.860,0.933
Dense (MiniLM),0.884,0.717,0.711
RRF BM25+TF-IDF,0.983,0.925,0.948
RRF BM25+Dense,0.945,0.867,0.876
RRF TF-IDF+Dense,0.950,0.852,0.902
Hybrid RRF (3 retriever),0.972,0.907,0.933



== R@10 (data uji, n = 3000) ==


variant,exact,partial,noisy
method,,,
BM25,0.997,0.983,0.965
TF-IDF karakter,0.995,0.948,0.981
Dense (MiniLM),0.945,0.841,0.814
RRF BM25+TF-IDF,0.997,0.981,0.984
RRF BM25+Dense,0.996,0.976,0.958
RRF TF-IDF+Dense,0.993,0.942,0.970
Hybrid RRF (3 retriever),0.997,0.978,0.981



== MRR (data uji, n = 3000) ==


variant,exact,partial,noisy
method,,,
BM25,0.988,0.957,0.935
TF-IDF karakter,0.978,0.892,0.950
Dense (MiniLM),0.907,0.760,0.749
RRF BM25+TF-IDF,0.989,0.947,0.962
RRF BM25+Dense,0.965,0.909,0.907
RRF TF-IDF+Dense,0.966,0.884,0.928
Hybrid RRF (3 retriever),0.983,0.934,0.952



== nDCG@10 (data uji, n = 3000) ==


variant,exact,partial,noisy
method,,,
BM25,0.990,0.963,0.942
TF-IDF karakter,0.982,0.904,0.957
Dense (MiniLM),0.915,0.778,0.763
RRF BM25+TF-IDF,0.991,0.955,0.967
RRF BM25+Dense,0.973,0.925,0.919
RRF TF-IDF+Dense,0.972,0.897,0.937
Hybrid RRF (3 retriever),0.986,0.944,0.959


In [8]:
# Uji berpasangan (hybrid vs pembanding)
sig = []
for v in VARIANTS:
    rr_h = 1.0 / RANKS[(HYBRID, v)]
    for name in METHODS:
        if name == HYBRID:
            continue
        d = rr_h - 1.0 / RANKS[(name, v)]
        lo, hi = boot_ci(d)
        p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        sig.append({"variant": v, "pembanding": name, "selisih_MRR": d.mean(), "CI95_lo": lo, "CI95_hi": hi, "p_wilcoxon": p})
sig = pd.DataFrame(sig)
sig["CI_tak_memuat_nol"] = (sig["CI95_lo"] > 0) | (sig["CI95_hi"] < 0)
sig.to_csv(f"{OUT_DIR}/significance.csv", index=False)
display(sig.round(4))

,variant,pembanding,selisih_MRR,CI95_lo,CI95_hi,p_wilcoxon,CI_tak_memuat_nol
0,exact,BM25,-0.0056,-0.0086,-0.0027,0.0002,True
1,exact,TF-IDF karakter,0.0044,0.0009,0.0081,0.0112,True
2,exact,Dense (MiniLM),0.0760,0.0678,0.0846,0.0000,True
3,exact,RRF BM25+TF-IDF,-0.0067,-0.0095,-0.0040,0.0000,True
4,exact,RRF BM25+Dense,0.0172,0.0138,0.0211,0.0000,True
5,exact,RRF TF-IDF+Dense,0.0166,0.0132,0.0202,0.0000,True
6,partial,BM25,-0.0235,-0.0285,-0.0185,0.0000,True
7,partial,TF-IDF karakter,0.0420,0.0354,0.0482,0.0000,True
8,partial,Dense (MiniLM),0.1732,0.1618,0.1848,0.0000,True
9,partial,RRF BM25+TF-IDF,-0.0132,-0.0172,-0.0091,0.0000,True


In [9]:
# Rincian perpanjang kueri dan analisis kegagalan
kl = Q.loc[test_idx, "k_lines"].to_numpy()
rows = []
for name in ["BM25", "TF-IDF karakter", "Dense (MiniLM)", HYBRID]:
    for v in VARIANTS:
        rr = 1.0 / RANKS[(name, v)]
        for k_ in sorted(set(kl)):
            rows.append({"method": name, "variant": v, "k_lines": int(k_), "n": int((kl == k_).sum()),
                         "MRR": rr[kl == k_].mean()})
by_lines = pd.DataFrame(rows)
by_lines.to_csv(f"{OUT_DIR}/by_lines.csv", index=False)
display(by_lines.pivot_table(index=["variant", "method"], columns="k_lines", values="MRR").round(3))

titles = pd.read_parquet(find("songs_clean.parquet"), columns=["song_id", "title", "artist"]).set_index("song_id")
fail = np.flatnonzero(RANKS[(HYBRID, "noisy")] > 10)
print(f"\nKegagalan Hybrid pada kueri berisik (rank > 10): {len(fail)} dari {len(test_idx)}")
for j in fail[:8]:
    qi = test_idx[j]
    top1 = fuse([cache[(r, 'noisy')][0][qi] for r in METHODS[HYBRID]], DEPTH, RRF_K)[:1]
    print(f"\n[{Q.qid[qi]}] {Q.k_lines[qi]} baris | target: {titles.title[target[qi]]} - {titles.artist[target[qi]]}")
    print("  kueri  :", Q.q_noisy[qi][:140])
    print("  peringkat target:", RANKS[(HYBRID, 'noisy')][j],
          "| top-1:", titles.title[top1[0]] if top1 else "-")

k_lines                               1      2      3      4
variant method                                              
exact   BM25                      0.945  0.994  0.998  0.998
        Dense (MiniLM)            0.596  0.937  0.977  0.989
        Hybrid RRF (3 retriever)  0.914  0.991  0.998  0.999
        TF-IDF karakter           0.890  0.992  0.997  0.998
noisy   BM25                      0.691  0.965  0.991  0.994
        Dense (MiniLM)            0.263  0.719  0.888  0.921
        Hybrid RRF (3 retriever)  0.756  0.981  0.995  0.996
        TF-IDF karakter           0.739  0.983  0.997  0.995
partial BM25                      0.800  0.980  0.991  0.993
        Dense (MiniLM)            0.326  0.735  0.869  0.930
        Hybrid RRF (3 retriever)  0.698  0.955  0.991  0.993
        TF-IDF karakter           0.553  0.910  0.975  0.989


Kegagalan Hybrid pada kueri berisik (rank > 10): 57 dari 3000

[test-00034] 1 baris | target: I Was a Fool To Let You Go - Barry Manilow
  kueri  : and i cant stadn the nose
  peringkat target: 15.0 | top-1: Im on

[test-00083] 1 baris | target: Greenland Bound - A. L. Lloyd
  kueri  : and the stars behgun to burn
  peringkat target: 14.0 | top-1: Montreal Screwjob

[test-00096] 1 baris | target: Love You in the Dark - Stephen Hunley
  kueri  : and for the rest of my knifwe
  peringkat target: 40.0 | top-1: Rest Of My Life Nicky Romero Remix

[test-00169] 1 baris | target: One Thing - Built to Spill
  kueri  : therse one ring youve got to remember
  peringkat target: 13.0 | top-1: Rings of Gold

[test-00223] 1 baris | target: Sugar Pills - Katastro
  kueri  : ive biz tayin up all night
  peringkat target: 13.0 | top-1: Alcoholics Unanimous

[test-00247] 1 baris | target: Disco Ball Festive Edit - Karen June Lawrence
  kueri  : make me feel ilke im live
  peringkat target: inf | top-1:

In [10]:
# Latensi dan ukuran indeks
sizes = {"bm25_csc.npz": find("bm25_csc.npz"), "tfidf_csc.npz": find("tfidf_csc.npz"),
         "chunk_emb_fp16.npy": find("chunk_emb_fp16.npy")}
size_mb = {k: round(os.path.getsize(p) / 1e6, 1) for k, p in sizes.items()}
print("Ukuran indeks (MB):", size_mb)

texts_lat = []
for v in VARIANTS:
    texts_lat += Q["q_" + v].iloc[test_idx[:100]].tolist()
for t in texts_lat[:10]:                          # pemanasan
    search_bm25(t, K_CAND); search_tfidf(t, K_CAND); dense_batch([t], bs=1)

lat = {k: [] for k in ["bm25", "tfidf", "dense", "rrf"]}
for t in texts_lat:
    t0 = time.perf_counter(); ib, _ = search_bm25(t, K_CAND);  lat["bm25"].append(1000 * (time.perf_counter() - t0))
    t0 = time.perf_counter(); it, _ = search_tfidf(t, K_CAND); lat["tfidf"].append(1000 * (time.perf_counter() - t0))
    t0 = time.perf_counter(); idd, _ = dense_batch([t], bs=1); lat["dense"].append(1000 * (time.perf_counter() - t0))
    t0 = time.perf_counter(); fuse([ib, it, idd[0]], DEPTH, RRF_K);  lat["rrf"].append(1000 * (time.perf_counter() - t0))
lat_df = pd.DataFrame(lat)
lat_df["total_berurutan"] = lat_df[["bm25", "tfidf", "dense", "rrf"]].sum(axis=1)
lat_df["total_paralel"] = lat_df[["bm25", "tfidf", "dense"]].max(axis=1) + lat_df["rrf"]
lat_summary = pd.DataFrame({"p50_ms": lat_df.quantile(.5), "p95_ms": lat_df.quantile(.95)}).round(1)
display(lat_summary)
with open(f"{OUT_DIR}/latency.json", "w") as f:
    json.dump({"n_queries": len(texts_lat), "size_mb": size_mb, "ms": lat_summary.to_dict()}, f, indent=2)

Ukuran indeks (MB): {'bm25_csc.npz': 254.5, 'tfidf_csc.npz': 3717.8, 'chunk_emb_fp16.npy': 1415.7}


,p50_ms,p95_ms
bm25,26.2,45.1
tfidf,182.9,374.5
dense,19.3,20.3
rrf,0.3,0.4
total_berurutan,228.9,431.4
total_paralel,183.2,374.8


In [11]:
# Demo alur daring (Algoritma 1)
chunk_text = pd.read_parquet(find("chunks_L3_S2.parquet"), columns=["text"])["text"].tolist()
LABEL = {"bm25": "Exact", "tfidf": "Fuzzy", "dense": "Semantic"}

def search_hybrid(q, top=10):
    """Alur daring (Algoritma 1): tiga retriever -> RRF -> agregasi max ke lagu -> label kecocokan."""
    ib, _ = search_bm25(q, K_CAND)
    it, _ = search_tfidf(q, K_CAND)
    idd, _ = dense_batch([q], bs=1)
    lists = {"bm25": ib, "tfidf": it, "dense": idd[0]}
    sc, rank_in = {}, {}
    for name, ids in lists.items():
        for r in range(min(DEPTH, len(ids))):
            c = int(ids[r])
            if c < 0:
                break
            sc[c] = sc.get(c, 0.0) + 1.0 / (RRF_K + r + 1)
            rank_in.setdefault(c, {})[name] = r + 1
    best = {}
    for c, s in sc.items():
        sid = int(song_of[c])
        if sid not in best or s > best[sid][0]:
            best[sid] = (s, c)
    out = []
    for pos, (sid, (s, c)) in enumerate(sorted(best.items(), key=lambda x: -x[1][0])[:top], 1):
        r = rank_in[c]
        out.append({"#": pos, "judul": titles.title[sid], "artis": titles.artist[sid], "skor": round(s, 4),
                    "cocok": LABEL[min(r, key=r.get)], "cuplikan": chunk_text[c].replace("\n", " / ")[:90]})
    return pd.DataFrame(out)

for q in ["i wanna hold your hand", "thinking about you every night alone", "weer all in this togethr"]:
    print("\nKueri:", q)
    display(search_hybrid(q, 5))


Kueri: i wanna hold your hand


,#,judul,artis,skor,cocok,cuplikan
0,1,I Want To Hold Your Hand Glee Cast Concert Ver...,Glee Cast,0.2727,Exact,I wanna hold your hand / I wanna hold your han...
1,2,I Want To Hold Your Hand,Mayday Parade,0.1662,Semantic,I want to hold your hand / I want to hold your...
2,3,Best Friends,XS,0.1610,Exact,"I wanna hold your hand, your hand / I wanna ho..."
3,4,Thanks For Being Material,Red Lipstick Letter,0.1104,Semantic,Can i hold your hand / I won't hold your hand ...
4,5,Wanted,Boyce Avenue,0.1027,Exact,I wanna make you feel wanted / And I wanna cal...



Kueri: thinking about you every night alone


,#,judul,artis,skor,cocok,cuplikan
0,1,Love Crazy,Karen Harding,0.1960,Fuzzy,Thinking about me / Thinking about you / And I
1,2,Johan Kim Chancellor - Beautiful English Tran...,Genius English Translations,0.1742,Exact,"I can't stop thinking about you / Every night,..."
2,3,Million Girl,Hacelen Royer,0.1461,Exact,And it feels so right / I've been thinking abo...
3,4,Just for Me,Al Green,0.1246,Fuzzy,Thinking about you everyday this way / Thinkin...
4,5,About You,G Flip,0.1207,Fuzzy,Here thinking about you / About you / (There t...



Kueri: weer all in this togethr


,#,judul,artis,skor,cocok,cuplikan
0,1,Family Julie Lov Remix,Drew Holcomb & The Neighbors,0.1534,Fuzzy,All in this together (Hey) / Hey / All in this...
1,2,Were All In This Together,Ben Lee,0.1310,Fuzzy,It's a big decision / But we're all in this to...
2,3,R.I.P. Foghorn,The Amity Affliction,0.1224,Fuzzy,We're all in this together / Shining bright th...
3,4,Whatever,Jermaine Dupri,0.0909,Exact,"Oooooh, homeboy T-I-G / Southside riders, Nate..."
4,5,Dominic,A Mankind Complex,0.0909,Semantic,But it doesn't matter / Now tell me why / (Bec...


In [12]:
# Simpan hasil
rows = []
for (name, v), r in RANKS.items():
    for j, qi in enumerate(test_idx):
        rows.append((Q.qid[qi], v, name, r[j] if np.isfinite(r[j]) else np.nan))
pd.DataFrame(rows, columns=["qid", "variant", "method", "rank"]).to_parquet(f"{OUT_DIR}/eval_per_query.parquet", index=False)
with open(f"{OUT_DIR}/final_config.json", "w") as f:
    json.dump({"rrf_k": RRF_K, "depth": DEPTH, "k_cand": K_CAND, "eval_top": EVAL_TOP,
               "variants": VARIANTS, "n_val": int(len(val_idx)), "n_test": int(len(test_idx)),
               "seed": SEED}, f, indent=2)
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"{fn:28s} {os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6:8.2f} MB")

__notebook__.ipynb               0.09 MB
by_lines.csv                     0.00 MB
eval_per_query.parquet           0.07 MB
final_config.json                0.00 MB
latency.json                     0.00 MB
results_test.csv                 0.00 MB
retrieval_cache.npz             28.80 MB
rrf_tuning_val.csv               0.00 MB
significance.csv                 0.00 MB


In [13]:
# RRF berbobot
def fuse_w(lists, depth, k, w):
    """RRF berbobot: skor(c) = sum_r w_r / (k + rank_r(c)); lalu agregasi max ke lagu."""
    sc = {}
    for ids, wt in zip(lists, w):
        if wt == 0:
            continue
        for r in range(min(depth, len(ids))):
            c = ids[r]
            if c < 0:
                break
            sc[c] = sc.get(c, 0.0) + wt / (k + r + 1)
    best = {}
    for c, s in sc.items():
        sid = song_of[c]
        if s > best.get(sid, 0.0):
            best[sid] = s
    return [sid for sid, _ in sorted(best.items(), key=lambda x: -x[1])[:EVAL_TOP]]

def eval_w(variant, rows, depth, k, w):
    out = np.empty(len(rows))
    for j, qi in enumerate(rows):
        lists = [cache[(r, variant)][0][qi] for r in ["bm25", "tfidf", "dense"]]
        out[j] = target_rank(fuse_w(lists, depth, k, w), target[qi])
    return out

# --- penyetelan pada data VALIDASI saja: k, kedalaman, dan bobot dense (bobot BM25 = TF-IDF = 1) ---
grid_k2, grid_depth2, grid_wd = [1, 3, 5, 10, 20, 60], [50, 100], [0.0, 0.25, 0.5, 1.0]
tune_w = []
for k in grid_k2:
    for depth in grid_depth2:
        for wd in grid_wd:
            row = {"k": k, "depth": depth, "w_dense": wd}
            for v in VARIANTS:
                row["MRR_" + v] = metrics(eval_w(v, val_idx, depth, k, (1.0, 1.0, wd)))["MRR"]
            row["MRR_rerata"] = np.mean([row["MRR_" + v] for v in VARIANTS])
            tune_w.append(row)
tune_w = pd.DataFrame(tune_w).sort_values("MRR_rerata", ascending=False).reset_index(drop=True)
display(tune_w.head(10).round(4))
print("Per bobot dense (MRR rerata terbaik untuk tiap bobot):")
display(tune_w.groupby("w_dense")["MRR_rerata"].max().round(4))
tune_w.to_csv(f"{OUT_DIR}/rrf_weighted_tuning_val.csv", index=False)

BK, BD, BW = int(tune_w.loc[0, "k"]), int(tune_w.loc[0, "depth"]), float(tune_w.loc[0, "w_dense"])
print(f"Dipilih dari validasi: k = {BK}, kedalaman = {BD}, bobot dense = {BW}")

# --- evaluasi SATU KALI pada data uji ---
NAME_W = "Hybrid RRF berbobot"
rows = []
for v in VARIANTS:
    r = eval_w(v, test_idx, BD, BK, (1.0, 1.0, BW))
    RANKS[(NAME_W, v)] = r
    lo, hi = boot_ci(1.0 / r)
    rows.append({"method": NAME_W, "variant": v, **metrics(r), "MRR_lo": lo, "MRR_hi": hi})
res_w = pd.DataFrame(rows)
display(res_w.round(3))

cmp_rows = []
for v in VARIANTS:
    for other in [HYBRID, "RRF BM25+TF-IDF", "BM25"]:
        d = 1.0 / RANKS[(NAME_W, v)] - 1.0 / RANKS[(other, v)]
        lo, hi = boot_ci(d)
        p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        cmp_rows.append({"variant": v, "pembanding": other, "selisih_MRR": d.mean(), "CI95_lo": lo, "CI95_hi": hi,
                         "p_wilcoxon": p, "CI_tak_memuat_nol": (lo > 0) or (hi < 0)})
cmp_w = pd.DataFrame(cmp_rows)
display(cmp_w.round(4))

kl = Q.loc[test_idx, "k_lines"].to_numpy()
tbl = []
for name in ["BM25", "RRF BM25+TF-IDF", HYBRID, NAME_W]:
    for v in VARIANTS:
        rr = 1.0 / RANKS[(name, v)]
        for k_ in sorted(set(kl)):
            tbl.append({"method": name, "variant": v, "k_lines": int(k_), "MRR": rr[kl == k_].mean()})
display(pd.DataFrame(tbl).pivot_table(index=["variant", "method"], columns="k_lines", values="MRR").round(3))

res_w.to_csv(f"{OUT_DIR}/results_test_weighted.csv", index=False)
cmp_w.to_csv(f"{OUT_DIR}/significance_weighted.csv", index=False)
with open(f"{OUT_DIR}/final_config_weighted.json", "w") as f:
    json.dump({"rrf_k": BK, "depth": BD, "w_bm25": 1.0, "w_tfidf": 1.0, "w_dense": BW}, f, indent=2)

,k,depth,w_dense,MRR_exact,MRR_partial,MRR_noisy,MRR_rerata
0,10,100,0.0,0.9912,0.9552,0.9602,0.9689
1,20,100,0.0,0.9912,0.9551,0.9602,0.9688
2,3,100,0.0,0.9917,0.9544,0.9598,0.9686
3,3,50,0.0,0.9917,0.9544,0.9598,0.9686
4,60,100,0.0,0.9911,0.9543,0.9604,0.9686
5,10,50,0.0,0.9912,0.9551,0.9595,0.9686
6,1,100,0.0,0.9902,0.9551,0.9604,0.9686
7,1,50,0.0,0.9902,0.9551,0.9604,0.9686
8,20,50,0.0,0.9912,0.9546,0.9593,0.9684
9,5,100,0.0,0.9913,0.9549,0.9589,0.9683


Per bobot dense (MRR rerata terbaik untuk tiap bobot):


w_dense
0.00    0.9689
0.25    0.9681
0.50    0.9677
1.00    0.9650
Name: MRR_rerata, dtype: float64

Dipilih dari validasi: k = 10, kedalaman = 100, bobot dense = 0.0


,method,variant,R@1,R@5,R@10,MRR,nDCG@10,MRR_lo,MRR_hi
0,Hybrid RRF berbobot,exact,0.983,0.996,0.997,0.989,0.991,0.986,0.992
1,Hybrid RRF berbobot,partial,0.925,0.972,0.981,0.947,0.955,0.940,0.953
2,Hybrid RRF berbobot,noisy,0.948,0.977,0.984,0.962,0.967,0.955,0.968


,variant,pembanding,selisih_MRR,CI95_lo,CI95_hi,p_wilcoxon,CI_tak_memuat_nol
0,exact,Hybrid RRF (3 retriever),0.0067,0.0039,0.0095,0.0000,True
1,exact,RRF BM25+TF-IDF,0.0000,0.0000,0.0000,1.0000,False
2,exact,BM25,0.0011,-0.0006,0.0028,0.1907,False
3,partial,Hybrid RRF (3 retriever),0.0132,0.0092,0.0177,0.0000,True
4,partial,RRF BM25+TF-IDF,0.0000,0.0000,0.0000,1.0000,False
5,partial,BM25,-0.0103,-0.0143,-0.0062,0.0000,True
6,noisy,Hybrid RRF (3 retriever),0.0097,0.0068,0.0126,0.0000,True
7,noisy,RRF BM25+TF-IDF,0.0000,0.0000,0.0000,1.0000,False
8,noisy,BM25,0.0262,0.0216,0.0310,0.0000,True


k_lines                               1      2      3      4
variant method                                              
exact   BM25                      0.945  0.994  0.998  0.998
        Hybrid RRF (3 retriever)  0.914  0.991  0.998  0.999
        Hybrid RRF berbobot       0.949  0.994  0.999  0.999
        RRF BM25+TF-IDF           0.949  0.994  0.999  0.999
noisy   BM25                      0.691  0.965  0.991  0.994
        Hybrid RRF (3 retriever)  0.756  0.981  0.995  0.996
        Hybrid RRF berbobot       0.797  0.989  0.997  0.997
        RRF BM25+TF-IDF           0.797  0.989  0.997  0.997
partial BM25                      0.800  0.980  0.991  0.993
        Hybrid RRF (3 retriever)  0.698  0.955  0.991  0.993
        Hybrid RRF berbobot       0.747  0.972  0.991  0.996
        RRF BM25+TF-IDF           0.747  0.972  0.991  0.996